# 03_context_progress

Stage 3 of the `codes/mcp/` track: **Context + progress** on the same
**FastMCP + Pydantic** stack as stages 1–2.

Notebook path: `codes/mcp/03_context_progress/qa.ipynb`. File I/O for this
stage stays in this directory (`server.py`, `client.py`). Shared deps live in
`codes/requirements.txt` (one file for the whole `codes/` lab tree).

Install once (venv recommended), from this stage directory:

```bash
pip install -r ../../requirements.txt
```

Run the next cell first to bind `ROOT`. Edit only under `# answer`. Pre-filled
setup code stays as-is. Get each cell running before moving on.

Architecture to keep in your head:

`Client(mcp, progress_handler=...) → call_tool` while the tool does
`await ctx.report_progress(progress, total, message=...)`

`Context` is injected by FastMCP and **does not** appear in the tool's
`input_schema`. Progress values are **absolute** (not deltas). Protocol-level
logging is deprecated in the `2026-07-28` era — this lab sticks to progress.

FastMCP's Client is async — cells below use top-level `await` (Jupyter /
IPython). From a plain script, wrap the same coroutine in `asyncio.run(...)`.


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """Directory that contains this qa.ipynb."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "03_context_progress" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "mcp" / "03_context_progress"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/mcp/03_context_progress")

ROOT = lab_root()
print(ROOT)


/workspace/AEFS-Notes/codes/mcp/03_context_progress


## 1. Read a host-facing tools/list shape (Context stays out)

`payload` is what a host sees after listing tools. Progress-capable tools still
advertise only their user-facing arguments — FastMCP strips `Context`.

Print every tool `name`, then print the property keys of `count_up`.
Confirm `ctx` is **not** among them.


In [2]:
payload = {
    "tools": [
        {
            "name": "count_up",
            "description": "Count 1..n, reporting absolute progress after each step.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "n": {
                        "description": "How many steps (1..5)",
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 5,
                    }
                },
                "required": ["n"],
            },
        }
    ]
}

# answer

names = [t["name"] for t in payload["tools"]]
print(names)
count = next(t for t in payload["tools"] if t["name"] == "count_up")
props = sorted(count["inputSchema"]["properties"])
print(props)
assert "ctx" not in props


['count_up']
['n']


## 2. Tiny FastMCP server that reports progress

Create a local `FastMCP("qa-prog")` with one async tool
`tick(steps: Annotated[int, Field(...)], ctx: Context) -> str` that:

1. loops `i = 1 .. steps`
2. `await ctx.report_progress(progress=float(i), total=float(steps), message=f"tick {i}")`
3. returns `f"ticks:{steps}"`

Drive it with `Client(server, progress_handler=...)` for `steps=3`. Print:

1. negotiated `protocol_version` (expect `2026-07-28`)
2. each progress triple `(progress, total, message)` your handler saw
3. the final tool text


In [3]:
from typing import Annotated

from fastmcp import Client, Context, FastMCP
from pydantic import Field

# answer

server = FastMCP("qa-prog")
seen: list[tuple[float, float | None, str | None]] = []

@server.tool
async def tick(
    steps: Annotated[int, Field(description="How many ticks", ge=1, le=5)],
    ctx: Context,
) -> str:
    """Emit absolute progress for each tick."""
    for i in range(1, steps + 1):
        await ctx.report_progress(
            progress=float(i),
            total=float(steps),
            message=f"tick {i}",
        )
    return f"ticks:{steps}"

async def on_progress(progress: float, total: float | None, message: str | None) -> None:
    seen.append((progress, total, message))

async with Client(server, progress_handler=on_progress) as client:
    print("protocol:", client.protocol_version)
    result = await client.call_tool("tick", {"steps": 3})
    text = result.content[0].text if result.content else result.data
    print("progress:", seen)
    print("result:", text)


protocol: 2026-07-28


progress: [(1.0, 3.0, 'tick 1'), (2.0, 3.0, 'tick 2'), (3.0, 3.0, 'tick 3')]
result: ticks:3


## 3. Drive the lab server: list → call with progress

Import `mcp` from `server.py` in this directory. With
`Client(mcp, progress_handler=...)`:

1. print `protocol_version` and `server_info.name`
2. `list_tools` — print names and `input_schema` property keys (no `ctx`)
3. `call_tool("count_up", {"n": 3})` — print each progress line and the result

Same Client entry as stages 1–2 — only the mid-call notification surface grows.


In [4]:
import sys

sys.path.insert(0, str(ROOT))
from fastmcp import Client
from server import mcp

# answer

lab_progress: list[tuple[float, float | None, str | None]] = []

async def lab_on_progress(
    progress: float, total: float | None, message: str | None
) -> None:
    lab_progress.append((progress, total, message))
    print("progress:", progress, total, message)

async with Client(mcp, progress_handler=lab_on_progress) as client:
    print("protocol:", client.protocol_version)
    print("server:", client.server_info.name if client.server_info else None)
    tools = await client.list_tools()
    print("tools:", [t.name for t in tools])
    print("properties:", sorted(tools[0].input_schema.get("properties", {})))
    result = await client.call_tool("count_up", {"n": 3})
    text = result.content[0].text if result.content else result.data
    print("result:", text)
    print("n_progress:", len(lab_progress))


protocol: 2026-07-28
server: lab-context-progress
tools: ['count_up']
properties: ['n']
progress: 1.0 3.0 step 1
progress: 2.0 3.0 step 2
progress: 3.0 3.0 step 3
result: done:3
n_progress: 3


## 4. Absolute progress (not deltas)

Build a local `FastMCP("qa-abs")` tool `fill(ctx: Context) -> str` that reports
progress at **25, then 50, then 100** out of `total=100` (absolute values).
Capture the handler's `progress` values and print them, then the result text
`filled`.

Extension point for later stages: keep adding ordinary typed callables;
transport / protocol stay with FastMCP. Prefer progress over deprecated
protocol logging in the `2026-07-28` era.


In [5]:
# answer

ext = FastMCP("qa-abs")
abs_seen: list[float] = []

@ext.tool
async def fill(ctx: Context) -> str:
    """Report absolute progress checkpoints."""
    for value in (25.0, 50.0, 100.0):
        await ctx.report_progress(progress=value, total=100.0, message=f"at {int(value)}")
    return "filled"

async def abs_on_progress(
    progress: float, total: float | None, message: str | None
) -> None:
    abs_seen.append(progress)

async with Client(ext, progress_handler=abs_on_progress) as client:
    result = await client.call_tool("fill", {})
    text = result.content[0].text if result.content else result.data
    print(abs_seen)
    print(text)


[25.0, 50.0, 100.0]
filled
